# 02 · Multi-resolution NF — grid refinement + bad-voxel filter

Multi-resolution mode kicks in whenever the param file has

    GridRefactor <StartingGridSize> <ScalingFactor> <NumLoops>

The pipeline runs an initial coarse pass at `StartingGridSize`, then **N**
refinement loops where:

1. The current `.mic` is clustered into grains (`Mic2GrainsList`).
2. A *seeded* fit at the refined grid uses those grains as seeds.
3. Voxels with `confidence < MinConfidence` are flagged as **bad**.
4. An *unseeded* fit re-runs only on the bad voxels.
5. The seeded + unseeded results are **binary-merged** (unseeded records
   overlay seeded records at the corresponding voxel offsets).
6. `ParseMic` runs on the merged binary.

Each loop writes a `multi_resolution/loop_<i>_<seeded|unseeded|merged>/` group
into the consolidated H5.

In [ ]:
import os, shutil, tempfile, time
from pathlib import Path
from argparse import Namespace

import h5py, numpy as np
import matplotlib.pyplot as plt

from midas_nf_pipeline.workflows import run_layer_pipeline

In [ ]:
MIDAS_HOME = Path(os.environ.get('MIDAS_HOME') or os.environ.get('MIDAS_INSTALL_DIR') or '.')
AU_DIR = MIDAS_HOME / 'NF_HEDM/Example/sim'

ws = Path(tempfile.mkdtemp(prefix='nf_multi_res_'))
param = ws / 'test_ps_au.txt'
shutil.copy2(AU_DIR / 'test_ps_au.txt', param)

# Add GridRefactor + SeedOrientationsAll for multi-res mode.
# StartingGridSize=2.0  ScalingFactor=2.0  NumLoops=2 → loop 0 @2.0, 1 @1.0, 2 @0.5
with open(param, 'a') as f:
    f.write('GridRefactor 2.0 2.0 2\n')
    f.write(f'SeedOrientationsAll {ws / "seedorients_full.csv"}\n')
    f.write('MinConfidence 0.5\n')
    f.write(f'OutputDirectory {ws}\n')
print('workspace:', ws)

In [ ]:
args = Namespace(
    paramFN=str(param), nCPUs=4, device='auto',
    ffSeedOrientations=False, doImageProcessing=1,
    startLayerNr=1, endLayerNr=1, resultFolder=str(ws),
    minConfidence=0.6, resume='', restartFrom='',
)
t0 = time.time()
h5 = run_layer_pipeline(args, install_dir=str(MIDAS_HOME))
print(f'pipeline finished in {time.time()-t0:.1f}s')
print('H5 →', h5)

## Inspect per-loop resolutions

In [ ]:
with h5py.File(h5, 'r') as fp:
    if 'multi_resolution' in fp:
        for label in fp['multi_resolution']:
            grp = fp[f'multi_resolution/{label}']
            attrs = dict(grp.attrs)
            print(label, attrs)

## Plot orientation maps from each loop side-by-side

You should see the resolution increase loop after loop, with the merged
result at the finest grid having the smoothest orientation field.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
with h5py.File(h5, 'r') as fp:
    for ax, label in zip(axes, ('loop_0_unseeded', 'loop_1_merged', 'loop_2_merged')):
        path = f'multi_resolution/{label}/maps/orientation'
        if path in fp:
            om = np.asarray(fp[path])
            ax.imshow(om[..., 0], origin='lower', cmap='viridis')
            ax.set_title(label)
        else:
            ax.set_title(f'{label} (missing)')
plt.tight_layout()
plt.savefig(ws / 'multires_progression.png', dpi=150)
print('plot:', ws / 'multires_progression.png')